# Deep Learning 074 — Scaled Dot-Product Attention (why `/ sqrt(d_k)`)

Companion notebook to the lesson. One division separates lesson 073's formula from the
published one, and the reason is a single measurable fact about dot products.

We reproduce all of it:

| Claim | Number |
|---|---|
| `Var(q·k) = d_k`, exactly | 513.85 measured at $d_k$ = 512 |
| Scaling restores it at every dimension | 1.00 |
| Unscaled softmax saturates | max weight 0.952, entropy 0.118 of 2.303 |
| The damage is **concentrated** | least key's gradient factor 3.9e-11 |
| Both ends of the range fail | entropy 2.296 of 2.303 when over-scaled |
| The divisor is per-head | 8, not 22.6 |

NumPy and matplotlib only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

## Part A — `Var(q·k) = d_k`

`Q @ K.T` looks like one matrix operation but it is a grid of scalars: with 3 queries
and 3 keys it is **9 separate dot products**. Nine numbers have a variance — and that
variance depends on something you would not expect it to.

A `d_k`-dimensional dot product is a **sum of `d_k` products**. The terms are
independent, and variances of independent terms **add**. So the variance of the sum
grows linearly in `d_k`. Measure it.

In [ ]:
SAMPLES = 200_000
print(f"{'d_k':>6}{'mean':>9}{'variance':>11}{'predicted':>11}{'std':>8}"
      f"{'after /sqrt(d_k)':>19}")
measured = []
for d_k in (1, 3, 64, 100, 512, 1000):
    q = rng.normal(size=(SAMPLES, d_k))
    k = rng.normal(size=(SAMPLES, d_k))
    dots = np.einsum('ij,ij->i', q, k)
    scaled = (dots / np.sqrt(d_k)).var()
    measured.append((d_k, dots.var(), scaled))
    print(f'{d_k:>6}{dots.mean():>9.3f}{dots.var():>11.2f}{d_k:>11}'
          f'{dots.std():>8.2f}{scaled:>19.4f}')

The variance tracks the dimension to two decimal places, and **the last column is the
whole derivation in one number.**

### The derivation, on one row of the score matrix

Treat each score as a draw from a random variable $X$.

- 1-dimensional vectors → the row has variance $\text{Var}(X)$
- 2-dimensional → each score is a sum of 2 products → $2\,\text{Var}(X)$
- $d$-dimensional → $d\,\text{Var}(X)$

We want it to stay at $\text{Var}(X)$ regardless of $d$. The scaling rule is
$\text{Var}(cX) = c^2\text{Var}(X)$, so pick $c = 1/\sqrt{d}$:

$$\text{Var}\!\left(\frac{X}{\sqrt{d}}\right) = \frac{1}{d} \cdot d\,\text{Var}(X) = \text{Var}(X)$$

**The square root exists to cancel a squaring.** Nothing in the argument is specific to
attention — it is a fact about sums of products.

In [ ]:
d_ks   = [m[0] for m in measured]
raw    = [m[1] for m in measured]
scaled = [m[2] for m in measured]

fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
ax[0].plot(d_ks, raw, 'o-', label='measured')
ax[0].plot(d_ks, d_ks, 'k--', lw=1, label='predicted = $d_k$')
ax[0].set(xlabel='$d_k$', ylabel='Var(q . k)', title='unscaled: variance IS the dimension')
ax[0].legend()
ax[1].plot(d_ks, scaled, 'o-', color='seagreen')
ax[1].axhline(1.0, color='k', ls='--', lw=1)
ax[1].set(xlabel='$d_k$', ylim=(0, 2), title='after dividing by sqrt($d_k$)')
plt.tight_layout(); plt.show()

> **Why not just use smaller vectors?** It would work, and it is the wrong trade. The
> whole point of a 512-dimensional embedding is that it holds more than a
> 3-dimensional one. Shrinking the representation to keep a variance in range pays for
> the fix with the thing the model is *for*. Scaling costs one division and changes no
> capacity at all.

## Part B — Why a wide spread is a problem: softmax is exponential

Easy to say, easy to underestimate. Two numbers.

In [ ]:
for row in ([1.0, 3.0], [1.0, 10.0], [1.0, 30.0]):
    p = softmax(np.array(row))
    print(f'softmax({row[0]:.0f}, {row[1]:>2.0f})  ->  {p[0]:.6f}, {p[1]:.6f}')

A gap of 2 is a **preference**. A gap of 29 is a **decision** — and there is no
gradient left on the losing side to ever revise it.

The standard deviation column in Part A says the gaps at `d_k = 512` are about
±22.7 wide before anything is scaled. So we are firmly in the second regime.

In [ ]:
N_KEYS, TRIALS = 10, 20_000
max_ent = np.log(N_KEYS)

def stats(p):
    ent = float(np.mean(-(p * np.log(p + 1e-300)).sum(1)))
    sens = float(np.mean(1.0 - (p ** 2).sum(1)))     # trace of the Jacobian
    p_min = p.min(1)
    return float(p.max(1).mean()), ent, sens, float(np.mean(p_min * (1 - p_min)))

print(f'{N_KEYS} keys, {TRIALS:,} rows. Maximum entropy = ln {N_KEYS} = {max_ent:.3f}\n')
print(f"{'d_k':>6}  {'--------- unscaled ---------':^36}  {'-------- / sqrt(d_k) --------':^36}")
print(f"{'':>6}  {'max p':>8}{'entropy':>9}{'1-sum p^2':>11}{'min grad':>10}"
      f"  {'max p':>8}{'entropy':>9}{'1-sum p^2':>11}{'min grad':>10}")
for d_k in (3, 64, 512, 1000):
    base = rng.normal(size=(TRIALS, N_KEYS)) * np.sqrt(d_k)
    r = stats(softmax(base))
    c = stats(softmax(base / np.sqrt(d_k)))
    print(f'{d_k:>6}  {r[0]:>8.3f}{r[1]:>9.3f}{r[2]:>11.4f}{r[3]:>10.1e}'
          f'  {c[0]:>8.3f}{c[1]:>9.3f}{c[2]:>11.4f}{c[3]:>10.1e}')

**Look at the right-hand block: it is flat.** Every statistic is identical at
`d_k = 3` and at `d_k = 1000`. That is what holding the variance at 1 achieves — *the
behaviour of the attention layer stops depending on how wide you made the model.*

Unscaled, growing the model silently changes what the softmax does, and not gently: by
`d_k = 512` it has stopped being a soft weighting at all.

## Part C — The gradient, which is where the real damage is

"The model is confident, so what?" The problem is what confidence does to *learning*.

The softmax Jacobian is $\text{diag}(p) - pp^\top$, and its diagonal entry
$p_i(1 - p_i)$ is **the factor multiplying every gradient that reaches the parameters
behind key $i$.** It is zero at both $p_i = 0$ and $p_i = 1$.

In [ ]:
p = np.linspace(1e-6, 1 - 1e-6, 400)
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(p, p * (1 - p), lw=2)
for val, name in [(0.952, 'max p, unscaled $d_k$=512'), (0.320, 'max p, scaled')]:
    ax.axvline(val, ls='--', lw=1, color='crimson' if val > 0.5 else 'seagreen')
    ax.text(val, 0.26, name, rotation=90, ha='right', va='top', fontsize=8)
ax.set(xlabel='$p_i$', ylabel='$p_i(1-p_i)$',
       title='the gradient gate: zero at BOTH ends')
plt.tight_layout(); plt.show()

In [ ]:
# Averages hide the problem. Compare the TOTAL sensitivity against the
# sensitivity of the LEAST attended key.
d_k = 512
base = rng.normal(size=(TRIALS, N_KEYS)) * np.sqrt(d_k)
r, c = stats(softmax(base)), stats(softmax(base / np.sqrt(d_k)))

print(f'at d_k = {d_k}')
print(f'  total sensitivity 1 - sum(p^2)   raw {r[2]:.4f}   scaled {c[2]:.4f}'
      f'   ratio {c[2]/r[2]:.0f}x')
print(f'  LEAST attended key p(1-p)        raw {r[3]:.1e}   scaled {c[3]:.1e}'
      f'   ratio {c[3]/r[3]:.0e}x')

**The failure is concentrated, not spread.** The total sensitivity falls by about 12×,
which sounds survivable. The *least attended* key's gradient factor falls by nine
orders of magnitude.

So it is not that learning slows down uniformly. It is that **the keys which lost the
argmax receive no gradient at all** — the model never discovers whether one of them
should have won. The winner keeps winning because it won.

> Note precisely which vanishing gradient this is. Lesson 060's version accumulated
> down a long chain of multiplications. **This one happens in a single layer**, from
> one saturated nonlinearity — which is the qualifier lesson 071's claim needs. The
> sequential-path version is genuinely gone; saturation is not, and `sqrt(d_k)` is why
> it does not bite.

## Part D — `sqrt(d_k)`, not merely *some* large divisor

If the aim were only to shrink the scores, any large number would do. It would not —
and the failure at the other end is the one lesson 073 already measured.

In [ ]:
d_k, d_model = 64, 512
base = rng.normal(size=(TRIALS, N_KEYS)) * np.sqrt(d_k)

print(f'd_k = {d_k} scores through {N_KEYS} keys, four divisors')
print(f"{'divisor':>22}{'max p':>8}{'entropy':>9}{'1-sum p^2':>11}{'min grad':>10}"
      f'   verdict')
rows = [('1  (unscaled)', 1.0, 'argmax, no gradient'),
        (f'sqrt(d_k) = {np.sqrt(d_k):.0f}', np.sqrt(d_k), "the paper's choice"),
        (f'sqrt(d_model) = {np.sqrt(d_model):.1f}', np.sqrt(d_model),
         'over-scaled, drifting to uniform'),
        (f'd_k = {d_k}', float(d_k), 'over-scaled, attention is an average')]
for name, c_div, verdict in rows:
    m, e, s, g = stats(softmax(base / c_div))
    print(f'{name:>22}{m:>8.3f}{e:>9.3f}{s:>11.4f}{g:>10.1e}   {verdict}')
print(f'\nmaximum possible entropy: {np.log(N_KEYS):.3f}')
print(f'sqrt(d_model)/sqrt(d_k) = {np.sqrt(d_model/d_k):.2g}x over-shrink')

In [ ]:
# Sweep the divisor continuously and watch both failures appear.
divisors = np.logspace(-0.2, 2.2, 60)
ents, mins = [], []
for c_div in divisors:
    _, e, _, g = stats(softmax(base / c_div))
    ents.append(e); mins.append(g)

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.4))
for a, vals, lab in [(ax[0], ents, 'entropy'), (ax[1], mins, 'least key $p(1-p)$')]:
    a.semilogx(divisors, vals, lw=2)
    a.axvline(np.sqrt(d_k), color='seagreen', ls='--', label='$\\sqrt{d_k}=8$')
    a.axvline(np.sqrt(d_model), color='crimson', ls=':', label='$\\sqrt{d_{model}}=22.6$')
    a.set(xlabel='divisor', ylabel=lab); a.legend(fontsize=8)
ax[0].axhline(np.log(N_KEYS), color='k', lw=0.8)
ax[0].set_title('too small -> argmax;  too large -> uniform')
ax[1].set_yscale('log'); ax[1].set_title('the gradient floor')
plt.tight_layout(); plt.show()

**Both ends fail, and they fail differently.**

- Too little scaling → an argmax with no gradient on the losing keys.
- Too much → entropy essentially at its maximum: a uniform average with no
  selectivity, which is **precisely the collapse measured in lesson 073**, reached from
  the opposite direction.

The parameterless model of lesson 073 was the over-scaled failure all along —
unit-norm embeddings put every score in `[-1, 1]`, which is what a very large divisor
would have produced anyway.

**And the divisor is per-head.** The 2017 base model has `d_model = 512` split across
8 heads, so the vectors actually being dotted inside a head are 64-dimensional and the
divisor is **8, not 22.6**. Using `sqrt(d_model)` would over-shrink by 2.83× and push
attention toward the uniform failure — the third row of the table measures the result.

## Part E — The complete mechanism

Put lesson 073 and lesson 074 together and you have the formula the whole field runs
on.

In [ ]:
def scaled_dot_product_attention(X, Wq, Wk, Wv):
    # The published formula, in five lines.
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    d_k = K.shape[-1]
    scores = (Q @ K.T) / np.sqrt(d_k)          # <-- the whole lesson
    weights = softmax(scores)
    return weights @ V, weights

n, d_model, d_k = 6, 64, 16
X = rng.normal(size=(n, d_model))
Wq, Wk, Wv = (rng.normal(size=(d_model, d_k)) / np.sqrt(d_model) for _ in range(3))
out, W = scaled_dot_product_attention(X, Wq, Wk, Wv)

print('output shape ', out.shape)
print('weight shape ', W.shape)
print('rows sum to 1', np.allclose(W.sum(1), 1))
print('\nattention weights\n', np.round(W, 3))

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Multi-head attention, next, changes nothing inside this formula — it runs several
copies of it at once.

## Exercises

1. **Non-Gaussian entries.** `Var(q·k) = d_k` was derived for unit-variance entries.
   Re-run Part A with uniform entries on $[-\sqrt{3}, \sqrt{3}]$ (also unit variance)
   and confirm the result is unchanged.
2. **Correlated vectors.** Make `q` and `k` correlated rather than independent. Does
   the variance still equal $d_k$? What does that imply for real trained models, where
   queries and keys are *not* independent?
3. **The temperature connection.** Dividing before a softmax is exactly the temperature
   knob used in text generation. Plot entropy against `1/temperature` and identify
   where `1/sqrt(d_k)` sits.
4. **Sequence length.** Every table here used 10 keys. Redo Part B with 2 keys and
   with 500. Does the saturation problem get better or worse with longer sequences,
   and why?
5. **Fix it a different way.** Instead of dividing, normalise `q` and `k` to unit
   length before the dot product (this is "cosine attention", used in some later
   models). Measure the variance and entropy. What does it cost you?